In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


<H1>Wandb(weights and biases) setup:</H1>

In [2]:
!pip install wandb

In [3]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
    #login to WandB using the key
    wandb.login(key=wandb_key)
    print("Successful connected with Weights & Biases!")
    
except Exception as e:
    print("Error, not logged in")
    print(f"Details: {e}")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Successful connected with Weights & Biases!


<h3>MODEL FROM SCRATCH, LSTM:</h3>

Importing basic stuff

In [4]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from transformers import get_cosine_schedule_with_warmup

setting up train and test

In [5]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

answer mapping to numbers:

In [6]:
answer_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(answer_mapping)

train-test-split for validation purposes(so that i do not have to use text.csv)

In [7]:
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42)

setting up the tokenizer

In [8]:
model_used = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_used)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Preprocessing: using the pytorch Dataset to duplicate 5 times plus padding 

In [9]:
class MCQDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        prompt = row['prompt']
        options = [row['A'], row['B'], row['C'], row['D'], row['E']]
        prompts = [prompt] * 5
        
        tokenized = self.tokenizer(
            prompts,
            options,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # FAANG FIX: Dynamically include token_type_ids if the tokenizer generates them
        output_dict = {
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'label': torch.tensor(row['label'], dtype=torch.long)
        }
        
        if 'token_type_ids' in tokenized:
            output_dict['token_type_ids'] = tokenized['token_type_ids']
            
        return output_dict

Dataset adn Dataloader

In [10]:
# Create dataset and dataloader
train_dataset = MCQDataset(train_data, tokenizer)
val_dataset = MCQDataset(val_data, tokenizer)

#experimenting with the batch_size rn
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

MAP@3 FUNCTION:

In [11]:
def calculate_map3(actual, predicted_top3):
    score = 0.0
    for a, p in zip(actual, predicted_top3):
        for i, pred in enumerate(p):
            if pred == a:
                score += 1.0 / (i + 1)
                break
    return score / len(actual)

#### Pretrained model - deberta-v3-small

In [12]:
#pre-trained model, initilaizing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForMultipleChoice.from_pretrained(model_used).to(device)
#using adam as the optimizer
optimizer = Adam(model.parameters(), lr=2e-5, eps=1e-6) #i used epsilon(eps) because i was getting training loss as "nan"

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias               

Training loop:

In [13]:
epochs = 3
accumulation_steps = 4

#total training steps = (number of batches / accumulation steps) * epochs
total_steps = (len(train_dataloader) // accumulation_steps) * epochs
#warmup for 10percent of total training time
warmup_steps = int(total_steps * 0.1)

classifier_params = []
base_params = []

for name, param in model.named_parameters():
    if "classifier" in name or "pooler" in name:
        classifier_params.append(param)
    else:
        base_params.append(param)

# The head learns 50x faster than the body to prevent catastrophic forgetting
optimizer = AdamW([
    {'params': base_params, 'lr': 1e-5},
    {'params': classifier_params, 'lr': 5e-4} 
], eps=1e-6)

In [14]:
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

In [15]:
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    model.train() 
    total_loss = 0

    optimizer.zero_grad()
    
    #tqdm for progress bar
    for step, batch in enumerate(tqdm(train_dataloader, desc="Training")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        kwargs = {
            'input_ids': input_ids, 
            'attention_mask': attention_mask, 
            'labels': labels
        }
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(device)
        
        #forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss / accumulation_steps
        total_loss += loss.item() * accumulation_steps
        
        #backward pass
        loss.backward()

        #to prevent gradients from exploding(cause i was getting training loss as "nan")
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        #optimizer step: Update weights
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_dataloader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step() # NEW: Step the learning rate scheduler
            optimizer.zero_grad()
        
    print(f"Average Training Loss: {total_loss / len(train_dataloader)}")

print("Training Complete!")

Epoch 1/3


Training: 100%|██████████| 400/400 [01:09<00:00,  5.74it/s]


Average Training Loss: 29.54986862182617
Epoch 2/3


Training: 100%|██████████| 400/400 [01:11<00:00,  5.60it/s]


Average Training Loss: 2.303966064453125
Epoch 3/3


Training: 100%|██████████| 400/400 [01:11<00:00,  5.63it/s]

Average Training Loss: 1.62968994140625
Training Complete!


Evaluation:

In [16]:
model.eval() 
all_actual = []
all_predicted = []
    
with torch.no_grad():
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
           
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            # The model outputs 'logits' (raw scores for each option)
            # We sort these scores in descending order to get the top predictions
            # argsort sorts ascending, so we reverse it with argsort(descending=True)
        top3_preds = torch.argsort(outputs.logits, dim=-1, descending=True)[:, :3]
         
        # Move data back to CPU to calculate the score
        all_actual.extend(labels.cpu().numpy())
        all_predicted.extend(top3_preds.cpu().numpy())
            
val_map3 = calculate_map3(all_actual, all_predicted)
print(f"Validation MAP@3 Score: {val_map3:.4f}")

print("\nPipeline Execution Complete!")

Validating: 100%|██████████| 100/100 [00:06<00:00, 14.31it/s]

Validation MAP@3 Score: 0.3979

Pipeline Execution Complete!


<hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr><hr>

<h1>For dummy submission initially</h1>

In [17]:
import pandas as pd

data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index = False)

<h1>Importing all dependencies required:</h1>

In [18]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

<h1>The following is for Milestone 1</h1>
<hr>
<h3>Question 1:</h3> Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [19]:
counts = df['answer'].value_counts()
sum_most_and_least = counts.max() + counts.min()
sum_most_and_least

814

<h3>Question 2:</h3> After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [20]:
def clean_text(text):
    text = text.lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)
vocab = set(' '.join(cleaned_prompts).split())
len(vocab)

859

<h3>Question 3:</h3> Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [21]:
row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)

13

<h3>Question 4:</h3> Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [22]:
combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())

2762

<h3>Question 5:</h3> Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [23]:
prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
option_a_vec = vectorizer.transform([df.iloc[0]['A']])
round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

np.float64(0.272)

<h3>Question 6:</h3> Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [24]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_v = vectorizer.transform([row['prompt']])
    scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100

np.float64(13.55)

<h3>Question 7:</h3> If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [25]:
gt_7 = 'C'
pred_7 = ['C', 'A', 'B']

score_7 = 1 / (pred_7.index(gt_7) + 1) if gt_7 in pred_7[:3] else 0.0

score_7

1.0

<h3>Question 8:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [26]:
gt_8 = 'B'
pred_8 = ['D', 'B', 'E']

score_8 = 1 / (pred_8.index(gt_8) + 1) if gt_8 in pred_8[:3] else 0.0
score_8

0.5

<h3>Question 9:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [27]:
top_3_answers = df['answer'].value_counts().index[:3].tolist()
baseline_map3 = df['answer'].apply(
    lambda ans: 1.0 if ans == top_3_answers[0] else 
               (0.5 if ans == top_3_answers[1] else 
               (1/3 if ans == top_3_answers[2] else 0.0))
).mean()
baseline_map3

np.float64(0.42125)

<h3>Question 10:</h3> The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [28]:
def map_at_3(truth, predictions):
    if truth in predictions[:3]:
        rank = predictions.index(truth) + 1
        return 1 / rank
    return 0


def get_top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    p_v = vectorizer.transform([row['prompt']])
    s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return sorted(s, key=s.get, reverse=True)

all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
all_scores.mean()

np.float64(0.2961666666666667)

<h1> MILESTONE 2:</h1>
<HR>


In [29]:
from datasets import load_dataset
from transformers import AutoTokenizer

file_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
dataset = load_dataset('csv', data_files=file_path)['train']

Generating train split: 0 examples [00:00, ? examples/s]

<h3>Question 1 - intro to hugging face transformers and models</h3> 
Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [30]:
def combine_text(example):
    example['combined_text'] = str(example['prompt']) + " " + str(example['A'])
    return example

dataset = dataset.map(combine_text)
combined_text_51 = dataset[51]['combined_text']
len(combined_text_51)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614

<h3>Question 2 - </h3>
Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [31]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer.vocab_size

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

<h3> Question 3 - </h3>
Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [32]:
#exact ID for the SEP token:
tokenizer.sep_token_id

102

<h3> Question 4 - </h3>
Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [33]:
prompts_list = [str(text) for text in dataset['prompt']]

tokens = tokenizer(prompts_list, padding='max_length', truncation=True, max_length=128, 
    return_tensors='pt')
tokens['input_ids'].shape


torch.Size([2000, 128])

<h3>Question 5 - BERT/RoBERTa architecture and attention mechanisms</h3>
A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  


In [34]:
import torch
from transformers import AutoModel

hidden_size = 768
attention_heads = 12
head_dim = hidden_size / attention_heads
print(int(head_dim))

64


<h3>Question 6 - </h3>
Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [35]:
model = AutoModel.from_pretrained('bert-base-uncased')
inputs_row_0 = tokenizer(dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs_row_0 = model(**inputs_row_0)
outputs_row_0.last_hidden_state.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

<h3>Question 7 - </h3>
Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [36]:
cls_vector = outputs_row_0.last_hidden_state[0, 0, :] # Batch 0, Token 0
sum_first_5 = cls_vector[:5].sum().item()
round(sum_first_5, 4)

-1.2001

<h3>Question 8 - </h3>
Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [37]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
test_string = "Light-ion fusion is a technique."
inputs_attn = tokenizer(test_string, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

tokens_list = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
fusion_index = tokens_list.index('fusion')

attention_matrix = outputs_attn.attentions[-1]
attention_weight = attention_matrix[0, 0, 0, fusion_index].item()
print(f"Q8 - Attention weight paid to 'fusion': {round(attention_weight, 4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q8 - Attention weight paid to 'fusion': 0.1025


<h3>Question 9 - context aware embeddings questions</h3>
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.


In [38]:
from sentence_transformers import SentenceTransformer, util

model_st = SentenceTransformer('all-MiniLM-L6-v2')

prompt_emb = model_st.encode(dataset[0]['prompt'])
option_b_emb = model_st.encode(dataset[0]['B'])

similarity = util.cos_sim(prompt_emb, option_b_emb).item()
round(similarity, 4)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658

<h3>Question 10 - </h3>
Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [39]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

sent_transformer_model = SentenceTransformer("all-MiniLM-L6-v2")

minilm_map3_score = []
specific_difference_count = 0
option_cols =['A', 'B', 'C', 'D', 'E']

for item in dataset:
    prompt = str(item['prompt'])
    options = [str(item[opt]) for opt in option_cols]
    correct_ans = item['answer']

    #the tf-idf pipeline
    vectorizer = TfidfVectorizer()
    docs = [prompt] + options
    tfidf_matrix = vectorizer.fit_transform(docs)

    tfidf_sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    tfidf_top3_index = tfidf_sims.argsort()[-3:][::-1]
    tfidf_top3_pred = [option_cols[i] for i in tfidf_top3_index]

    #second pipeline for allMiniLM-L6-v2
    prompt_embed = model_st.encode(prompt, convert_to_tensor = True)
    options_embed = model_st.encode(options, convert_to_tensor = True)
    minilm_sims = util.cos_sim(prompt_embed, options_embed)[0].cpu().numpy()

    minilm_top3_index = minilm_sims.argsort()[-3:][::-1]
    minilm_top3_pred = [option_cols[i] for i in minilm_top3_index]

    #comparing
    if correct_ans in minilm_top3_pred:
        rank = minilm_top3_pred.index(correct_ans) + 1
        minilm_map3_score.append(1.0/rank)
    else:
        minilm_map3_score.append(0.0)

    if (correct_ans not in tfidf_top3_pred) and (correct_ans in minilm_top3_pred):
        specific_difference_count += 1


#final calculation
final_map3_score = np.mean(minilm_map3_score)
print(f"Final map@3 score of the all-MiniLM-L6-v2 pipline is: {final_map3_score:.4f}")
print(f"Count of questions meeting the conditions: {specific_difference_count}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final map@3 score of the all-MiniLM-L6-v2 pipline is: 0.4231
Count of questions meeting the conditions: 564


In [40]:
574

574

<h3>Questions 11 - zero shot classification starts here</h3>
Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [41]:
from transformers import pipeline

#setting up zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

prompt_idx_1 = dataset[1]['prompt']
labels_idx_1 = [str(dataset[1]['A']), str(dataset[1]['B']), str(dataset[1]['C'])]

# ques11, zero-shot classification , softmax- probability sum to 1
result_softmax = classifier(prompt_idx_1, candidate_labels=labels_idx_1)
top_score = result_softmax['scores'][0]
round(top_score, 4)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4575

<h3>Question 12 - </h3>
Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [42]:
result_sigmoid = classifier(prompt_idx_1, candidate_labels=labels_idx_1, multi_label=True)

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])
absolute_difference = abs(sum_softmax - sum_sigmoid)

round(absolute_difference, 4)

0.9995

<h3>Question 13 - </h3>
Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [43]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt_str = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."

input_ids = tokenizer_t5(prompt_str, return_tensors="pt").input_ids
generated_tokens = model_t5.generate(input_ids, max_new_tokens=5)

exact_output = tokenizer_t5.decode(generated_tokens[0], skip_special_tokens=True)

exact_output

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

'B'

<h1>Milestone 3:</h1>
<hr>

<h4>Inital code to run(given in the milestone-3 google form)</h4>

In [44]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")
#Zero-shot classifier for Q1, Q2, Q6
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.5 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base successfully created


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

<h3>Question-1</h3>
Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [45]:
result_150 = zs(prompt_150, candidate_labels=labels_150)
correct_idx = result_150['labels'].index(ans_150)

prob_score_q1 = round(result_150['scores'][correct_idx], 3)
prob_score_q1

0.384

<h3>Question-2</h3>
Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [46]:
prompt_emb_150 = model.encode([prompt_150])
distances, indices = index.search(prompt_emb_150, k=10)

retrieved_indices = indices[0].tolist()
true_rank_faiss = retrieved_indices.index(150) + 1
true_rank_faiss

10

<h3>(The two stage pipeline: Reranking and cross-encoders)</h3>
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face)
Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

<h3>Question-3</h3>
Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [47]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

scored_docs = list(zip(ce_scores, retrieved_indices))
scored_docs.sort(key=lambda x: x[0], reverse=True)

sorted_indices = [doc[1] for doc in scored_docs]
true_rank_ce = sorted_indices.index(150) + 1
true_rank_ce

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1

<h3>Question-4</h3>
Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

<h3>Question-5</h3>
Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).


In [48]:
perfect_rag_string = f"Context: {kb[150]} Question: {prompt_150}"

result_perfect_rag = zs(perfect_rag_string, candidate_labels=labels_150)

correct_idx_perf = result_perfect_rag['labels'].index(ans_150)
prob_score_q5 = round(result_perfect_rag['scores'][correct_idx_perf], 3)
prob_score_q5

0.989

<h3>Concept: The Danger of Bad Retrieval (Adversarial RAG)</h3>
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

<h3>Question-6</h3>
What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).
</h3>

In [49]:
bad_rag_string = f"Context: {kb[999]} Question: {prompt_150}"

result_bad_rag = zs(bad_rag_string, candidate_labels=labels_150)

correct_idx_bad = result_bad_rag['labels'].index(ans_150)
prob_score_q6 = round(result_bad_rag['scores'][correct_idx_bad], 3)
prob_score_q6

0.529

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

<h3>Question-7</h3> For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [50]:
hits = 0
total_rows = 100

for idx in range(total_rows):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_ans_string = str(row[row['answer']])

    emb = model.encode([prompt])
    _, faiss_indices = index.search(emb, k=5)

    retrieved_docs = [kb[i] for i in faiss_indices[0]]

    if correct_ans_string in retrieved_docs:
        hits += 1

hit_rate = round((hits / total_rows) * 100, 1)
hit_rate

73.0

<h3>Question-8</h3>
Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).


In [51]:
def map_at_3(actual, preds):
    if actual == preds[0]: return 1.0
    elif actual == preds[1]: return 0.5
    elif actual == preds[2]: return 1/3
    else: return 0.0

map3_scores = []

for idx in range(20):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_label = row['answer']
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    labels_map = {str(row['A']): 'A', str(row['B']): 'B', str(row['C']): 'C', str(row['D']): 'D', str(row['E']): 'E'}
    

    emb = model.encode([prompt])
    _, faiss_indices = index.search(emb, k=5)
    top_5_docs = [kb[i] for i in faiss_indices[0]]
    
    pairs = [[prompt, doc] for doc in top_5_docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc_idx = np.argmax(ce_scores)
    best_document = top_5_docs[best_doc_idx]
    

    rag_str = f"Context: {best_document} Question: {prompt}"
    
    result = zs(rag_str, candidate_labels=options)
    
    top_3_preds = [labels_map[result['labels'][0]], 
                   labels_map[result['labels'][1]], 
                   labels_map[result['labels'][2]]]
    
    row_map3 = map_at_3(correct_label, top_3_preds)
    map3_scores.append(row_map3)

final_map3 = round(np.mean(map3_scores), 3)
final_map3

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


np.float64(0.975)

<h1>Milestone - </h1>
<hr>
Downloading required libraries

In [52]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

<h3>Question-1:</h3>
Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [53]:
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

df['label'] = df['answer'].map(label_map)

label_150 = df.loc[150, 'label']
label_150

np.int64(2)

<h3>Question-2:</h3>
Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?
*
Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

In [54]:
row_0 = df.iloc[0]
prompt_text = str(row_0['prompt'])
option_b_text = str(row_0['B'])
formatted_string = prompt_text + " [SEP] " + option_b_text
char_length = len(formatted_string)
char_length

407

<h3>Question-3:</h3>
Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [55]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

prompt_0 = str(df.iloc[0]['prompt'])
choices_0 = [str(df.iloc[0][opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
prompts_0 = [prompt_0] * 5 # Repeat prompt 5 times

tokens_0 = tokenizer(
    prompts_0, 
    choices_0, 
    padding="max_length", 
    truncation=True, 
    max_length=128, 
    return_tensors="pt"
)

input_ids_3d = tokens_0['input_ids'].unsqueeze(0)

second_dimension = input_ids_3d.shape[1]
second_dimension

5

<h3>Question 4:</h3>
Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?
*
Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

In [56]:
batch_size = 16
num_choices = 5
seq_length = 128

total_tokens = batch_size * num_choices * seq_length
total_tokens

10240

<h3>Questiom 5:</h3>
Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [57]:
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

with torch.no_grad():
    outputs = model(input_ids=input_ids_3d, attention_mask=tokens_0['attention_mask'].unsqueeze(0))

num_logits = outputs.logits.shape[1]
num_logits

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5

<h3>Question 6:</h3>
Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?
*
LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

In [58]:
label_0_tensor = torch.tensor([df.iloc[0]['label']])

with torch.no_grad():
    outputs_with_loss = model(input_ids=input_ids_3d, attention_mask=tokens_0['attention_mask'].unsqueeze(0),labels=label_0_tensor)

loss_tensor = outputs_with_loss.loss
num_dimensions = loss_tensor.dim()
num_dimensions

0

<h3>Question-7:</h3>
LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?
*
Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

In [59]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS 
)

peft_model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
trainable_params

295681

<h3>Question-8:</h3>
Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?
*
Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

In [60]:
df_100 = df.head(100).copy()

def preprocess_function(examples):
    prompts = [[context] * 5 for context in examples['prompt']]
    choices = []
    for i in range(len(examples['prompt'])):
        choices.append([examples['A'][i], examples['B'][i], examples['C'][i], examples['D'][i], examples['E'][i]])
    
    flat_prompts = sum(prompts, [])
    flat_choices = sum(choices, [])
    
    tokenized = tokenizer(flat_prompts, flat_choices, padding="max_length", truncation=True, max_length=128)
    
    results = {
        'input_ids': [tokenized['input_ids'][i : i + 5] for i in range(0, len(flat_prompts), 5)],
        'attention_mask': [tokenized['attention_mask'][i : i + 5] for i in range(0, len(flat_prompts), 5)],
        'labels': examples['label']
    }
    return results

hf_dataset = Dataset.from_pandas(df_100[['prompt', 'A', 'B', 'C', 'D', 'E', 'label']])
processed_dataset = hf_dataset.map(preprocess_function, batched=True, remove_columns=hf_dataset.column_names)

choices_stored = len(processed_dataset[0]['input_ids'])
choices_stored

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

5

<h3>Question-9:</h3>
Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [61]:
df_32 = df.head(32).copy()
hf_dataset_32 = Dataset.from_pandas(df_32[['prompt', 'A', 'B', 'C', 'D', 'E', 'label']])

def preprocess_64(examples):
    prompts = [[context] * 5 for context in examples['prompt']]
    choices = [[examples['A'][i], examples['B'][i], examples['C'][i], examples['D'][i], examples['E'][i]] for i in range(len(examples['prompt']))]
    
    tokenized = tokenizer(sum(prompts, []), sum(choices, []), padding="max_length", truncation=True, max_length=64)
    
    return {
        'input_ids': [tokenized['input_ids'][i:i+5] for i in range(0, len(tokenized['input_ids']), 5)],
        'attention_mask': [tokenized['attention_mask'][i:i+5] for i in range(0, len(tokenized['attention_mask']), 5)],
        'labels': examples['label']
    }

train_dataset_32 = hf_dataset_32.map(preprocess_64, batched=True, remove_columns=hf_dataset_32.column_names)

training_args = TrainingArguments(output_dir="./results",per_device_train_batch_size=4,gradient_accumulation_steps=1,max_steps=4,       logging_steps=1,report_to="none"    )

trainer = Trainer(model=peft_model,args=training_args,train_dataset=train_dataset_32,)

# 4. Execute fine-tuning and fetch final step
train_result = trainer.train()
train_result.global_step

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.205103
2,3.227842
3,3.227429
4,3.262107


4

<h3>Question-10:</h3>
Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [62]:
import torch.nn.functional as F
input_ids_0 = torch.tensor([train_dataset_32[0]['input_ids']]).to(peft_model.device)
attention_mask_0 = torch.tensor([train_dataset_32[0]['attention_mask']]).to(peft_model.device)

peft_model.eval()

with torch.no_grad():
    outputs = peft_model(input_ids=input_ids_0, attention_mask=attention_mask_0)

probabilities = F.softmax(outputs.logits, dim=-1)

prob_E = probabilities[0][4].item()
prob_E

0.20396263897418976